#  MIMIC-4 TIME SERIES PIPELINE PROJECT

*Author: Matthieu Komorowski 10/07/24*

This notebook enables the creation of a tabular format time series dataset in MIMIC-4

- Initiate the **connection to the MIMIC-IV database**
- **Extract the raw dataset**. This will require many steps:
    - Define the cohort of interest
    - Define the time period of interest and sampling frequency
    - Compute/extract the features of interest
    - Combine all those values in a single tabular format DataFrame


# Import required libraries and connect to the MIMIC-IV database

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pprint
import os

pd.set_option('display.max_colwidth',100000) #https://stackoverflow.com/questions/54692405/output-truncation-in-google-colab

# Below imports are used to print out pretty pandas dataframes
from IPython.display import display, HTML

# Imports for accessing data using Google BigQuery.
from google.colab import auth
from google.cloud import bigquery
from google.colab import files

In [2]:
#user authentication
auth.authenticate_user()

In [3]:
# Set up the project ID
# Note that the physionet-data project is for data hosting only.
project_id = 'rlpipeline-1'
os.environ['RLpipeline'] = project_id

In [4]:
# Read data from BigQuery into pandas dataframes.
def run_query(query):
  return pd.io.gbq.read_gbq(
      query,
      project_id=project_id,
      configuration={'query': {
          'useLegacySql': False
      }})

---

# Define the cohort of interest

Here we're interested in all adult patients with suspected infection, and will limit the cohort to their first ever ICU admission.
A feature called 'left_window' is created, which is the left bound of time period of interest. Here, we want to know the latest time among (ICU admission, suspected time of infection) for the sepsis cohort, so we can extract data later for the -24h -> +48h period.

In [5]:
query ="""

WITH t1 as(

SELECT
    ie.subject_id,
    ie.hadm_id,
    ie.stay_id,
    adm.admittime AS hospital_admittime,
    ie.intime AS icu_admittime,
    pat.anchor_age AS age,
    DENSE_RANK() OVER (PARTITION BY adm.subject_id ORDER BY ie.intime) AS icustay_seq,
    soi.suspected_infection_time
FROM
    `physionet-data.mimiciv_icu.icustays` ie
INNER JOIN
    `physionet-data.mimiciv_hosp.admissions` adm
    ON ie.hadm_id = adm.hadm_id
INNER JOIN
    `physionet-data.mimiciv_hosp.patients` pat
    ON ie.subject_id = pat.subject_id
LEFT JOIN (
    SELECT
        ie.subject_id,
        ie.stay_id,
        MIN(soi.suspected_infection_time) AS suspected_infection_time
    FROM
        `physionet-data.mimiciv_icu.icustays` ie
    LEFT JOIN
        `physionet-data.mimiciv_derived.suspicion_of_infection` soi
        ON ie.stay_id = soi.stay_id
    GROUP BY
        ie.subject_id,
        ie.stay_id
) soi
ON ie.subject_id = soi.subject_id AND ie.stay_id = soi.stay_id

)

SELECT subject_id, hadm_id, stay_id, icu_admittime, age, suspected_infection_time,
(case when t1.icu_admittime>suspected_infection_time then icu_admittime else suspected_infection_time end) as left_window
FROM t1
WHERE age >=18 and icustay_seq=1 and suspected_infection_time is not null
ORDER BY subject_id
LIMIT 10
"""

cohort = run_query(query)
cohort.head()

GenericGBQException: Reason: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/rlpipeline-1/jobs?prettyPrint=false: Access Denied: Project rlpipeline-1: User does not have bigquery.jobs.create permission in project rlpipeline-1.

Location: None
Job ID: 4f7d95fb-a27f-4dda-a08e-6efb1e00f596


# SAVE THE COHORT IN PERSONAL BIGQUERY TABLE

Run the SQL query in Big Query and then click on 'Save Results' / 'BigQuery table'

I saved it under rlpipeline-1.rlpipeline.cohort1


In [6]:
#define personal table containing cohort details
cohort1='`rlpipeline-1.rlpipeline.cohort1`'

#define time period of interest for the time series (in hours)
before_stamp=-24
after_stamp=48

#define time resolution (how many hours are summarised in each row)
time_resolution=1

---

# Vital signs

In [7]:
# Extract vital signs data from chartevents

# Define the item_id of interest for vital signs
item_ids = [220045,220277,220210, 225312,224322,220052,220181]
item_ids_str = ', '.join(str(item_id) for item_id in item_ids)

# SQL query to extract chartevents data for the given icustay_ids
query = f"""
WITH t1 as (
SELECT
    ce.hadm_id, ce.stay_id, cohort.left_window, floor(TIMESTAMP_DIFF(ce.chartTime, cohort.left_window , HOUR)/{time_resolution}) as offset, charttime, itemid, valuenum
FROM
   `physionet-data.mimiciv_icu.chartevents` ce
    left join  {cohort1} cohort
    on ce.stay_id = cohort.stay_id
WHERE
  itemid in ({item_ids_str}) and ce.stay_id in (select stay_id from  {cohort1})
limit 1000
), t2 as (
select * from t1
where offset >={before_stamp/time_resolution} and offset <={after_stamp/time_resolution}
order by stay_id, charttime
)

select distinct stay_id, max(charttime) as chartTime, offset,
avg(case when itemid in (220045) then valuenum else null end) as HR,
avg(case when itemid in (225312, 224322,220052, 220181) then valuenum else null end) as MBP,
avg(case when itemid in (220277) then valuenum else null end) as SpO2,
avg(case when itemid in (220210) then valuenum else null end) as RR
from t2
group by stay_id, offset
order by stay_id, offset
"""

# Run the query
vitals = run_query(query)

# Convert the result to a DataFrame
#chartevents_df = query_job.to_dataframe()

# Display the result
vitals.head(1000)

GenericGBQException: Reason: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/rlpipeline-1/jobs?prettyPrint=false: Access Denied: Project rlpipeline-1: User does not have bigquery.jobs.create permission in project rlpipeline-1.

Location: None
Job ID: dd477265-ee0b-4c8f-854d-6b874aa143cd


# Laboratory measurements



In [ ]:
# Extract lab data from labevents

# Define the item_id of interest for vital signs
item_ids = [50983, 51222]
item_ids_str = ', '.join(str(item_id) for item_id in item_ids)

# SQL query to extract chartevents data for the given icustay_ids
query = f"""
WITH t1 as (
SELECT
    le.hadm_id, cohort.stay_id, cohort.left_window, floor(TIMESTAMP_DIFF(le.chartTime, cohort.left_window , HOUR)/{time_resolution}) as offset, charttime, itemid, valuenum
FROM
   `physionet-data.mimiciv_hosp.labevents` le
    left join  {cohort1} cohort
    on le.hadm_id = cohort.hadm_id
WHERE
  itemid in ({item_ids_str}) and le.hadm_id in (select hadm_id from  {cohort1})
limit 1000
), t2 as (
select * from t1
where offset >={before_stamp/time_resolution} and offset <={after_stamp/time_resolution}
order by stay_id, charttime
)

select distinct stay_id, max(charttime) as chartTime, offset,
avg(case when itemid in (50983) then valuenum else null end) as Na,
avg(case when itemid in (51222) then valuenum else null end) as Hb
from t2
group by stay_id, offset
order by stay_id, offset
"""

# Run the query
labs = run_query(query)

# Convert the result to a DataFrame
#chartevents_df = query_job.to_dataframe()

# Display the result
labs.head(100)


,stay_id,chartTime,offset,Na,Hb
0,32610785,2112-12-03 06:36:00,-7.0,140.0,12.6
1,32610785,2112-12-04 06:50:00,17.0,140.0,12.0
2,32610785,2112-12-05 02:53:00,37.0,138.0,10.7
3,33685454,2129-08-04 17:56:00,0.0,139.0,12.5
4,33685454,2129-08-05 05:01:00,12.0,140.0,12.1
5,33685454,2129-08-06 06:05:00,37.0,141.0,10.4
6,33987268,2156-04-12 19:24:00,3.0,129.0,9.3
7,33987268,2156-04-13 01:41:00,9.0,128.0,10.2
8,33987268,2156-04-13 13:50:00,21.0,133.0,NaN
9,33987268,2156-04-14 04:38:00,36.0,130.0,11.8


# Urine output



### Exercise: try to list all the item_ids related to urine output in d_items

In [ ]:
# uo includes sum of hourly urine output ; cum_UO includes urine output volumes from ICU admission

query = f"""
WITH uo AS (
    SELECT
        oe.stay_id, oe.charttime, floor(TIMESTAMP_DIFF(oe.chartTime, cohort.left_window , HOUR)/{time_resolution}) as offset, CASE
            WHEN oe.itemid = 227488 AND oe.value > 0 THEN -1 * oe.value         -- note we consider input of GU irrigant as a negative volume.GU irrigant volume in usually has a corresponding volume out so the net is often 0, despite large irrigant volumes
            ELSE oe.value END AS urineoutput
    FROM `physionet-data.mimiciv_icu.outputevents` oe
    left join  {cohort1} cohort ON oe.stay_id = cohort.stay_id
WHERE
  oe.stay_id in (select stay_id from  {cohort1}) AND
  itemid IN
        (
            226559 -- Foley
            , 226560 -- Void
            , 226561 -- Condom Cath
            , 226584 -- Ileoconduit
            , 226563 -- Suprapubic
            , 226564 -- R Nephrostomy
            , 226565 -- L Nephrostomy
            , 226567 -- Straight Cath
            , 226557 -- R Ureteral Stent
            , 226558 -- L Ureteral Stent
            , 227488 -- GU Irrigant Volume In
            , 227489  -- GU Irrigant/Urine Volume Out
        )
), uo2 as (
SELECT
    stay_id
    , offset
    , SUM(urineoutput) AS uo
FROM uo
GROUP BY stay_id, offset), uo3 as (
select *,
sum(uo) over (partition by uo2.stay_id order by uo2.offset) as cum_uo
from uo2)

SELECT *
from uo3
where offset >={before_stamp/time_resolution} and offset <={after_stamp/time_resolution}
order by stay_id, offset

;
"""
# Run the query
uo = run_query(query)

# Display the result
uo.head(100)



,stay_id,offset,uo,cum_uo
0,32610785,-21.0,100.0,475.0
1,32610785,-13.0,100.0,575.0
2,32610785,-6.0,200.0,775.0
3,32610785,-1.0,150.0,925.0
4,32610785,1.0,150.0,1075.0
...,...,...,...,...
95,37510196,16.0,75.0,1175.0
96,37510196,17.0,30.0,1205.0
97,37510196,18.0,35.0,1240.0
98,37510196,19.0,30.0,1270.0


In [ ]:
uo.dtypes


stay_id      Int64
offset     float64
uo         float64
cum_uo     float64
dtype: object

# ICD codes

The international classification of diseases (ICD) is a coding system used to record patients' past medical history, diagnoses and procedures.
MIMIC-III contains ICD (version 9) data. This can be used to augment a mortality prediction score. For example, APACHE-IV includes several items related to patient past medical history: https://intensivecarenetwork.com/Calculators/Files/Apache4.html

Example: let's look for ICD codes for 'congestive heart failure'. You'll find these codes online, e.g.: https://en.wikipedia.org/wiki/List_of_ICD-9_codes_390%E2%80%93459:_diseases_of_the_circulatory_system#Other_forms_of_heart_disease_(420%E2%80%93429)

In [ ]:
query="""
select hadm_id, seq_num, icd9_code
, CASE
  when icd9_code in ('39891','40201','40211','40291','40401','40403','40411','40413','40491','40493') then 1
  when SUBSTR(icd9_code, 1, 4) in ('4254','4255','4257','4258','4259') then 1
  when SUBSTR(icd9_code, 1, 3) in ('428') then 1
  else 0 end as chf
from `mindstream-ai.mimiciii_clinical.diagnoses_icd`
where subject_id<100 -- to save time

"""

d  = run_query(query)
d.head(20)

,hadm_id,seq_num,icd9_code,chf
0,163353,1,V3001,0
1,163353,2,V053,0
2,163353,3,V290,0
3,145834,1,0389,0
4,145834,2,78559,0
5,145834,3,5849,0
6,145834,4,4275,0
7,145834,5,41071,0
8,145834,6,4280,1
9,145834,7,6826,0


We've already found some patients with a history of CHF!

Now let's look for some more ICD codes that we think could be related to mortality, and encode their presence as a binary feature. Our new 'COMORBIDITY' score can take values from 0 to 9.

In [ ]:
query="""
with icd as
(
  select hadm_id, seq_num, icd9_code
  from `mindstream-ai.mimiciii_clinical.diagnoses_icd`
  where seq_num != 1 -- we do not include the primary icd-9 code
)

, flag as
(
select hadm_id, icd9_code
, CASE
  when icd9_code in ('39891','40201','40211','40291','40401','40403','40411','40413','40491','40493') then 1
  when SUBSTR(icd9_code, 1, 4)  in ('4254','4255','4257','4258','4259') then 1
  when SUBSTR(icd9_code, 1, 3)  in ('428') then 1
  else 0 end as chf       /* Congestive heart failure */

, CASE
  when SUBSTR(icd9_code, 1, 4)  in ('4168','4169','5064','5081','5088') then 1
  when SUBSTR(icd9_code, 1, 3)  in ('490','491','492','493','494','495','496','500','501','502','503','504','505') then 1
  else 0 end as lung  /* Chronic pulmonary disease */

, CASE
  when SUBSTR(icd9_code, 1, 4) in ('2500','2501','2502','2503','2504','2505','2506','2507','2508','2509') then 1
  else 0 end as dm      /* Diabetes*/

, CASE
  when icd9_code in ('40301','40311','40391','40402','40403','40412','40413','40492','40493') then 1
  when SUBSTR(icd9_code, 1, 4)  in ('5880','V420','V451') then 1
  when SUBSTR(icd9_code, 1, 3)  in ('585','586','V56') then 1
  else 0 end as ren  /* Renal failure */

, CASE
  when icd9_code in ('07022','07023','07032','07033','07044','07054') then 1
  when SUBSTR(icd9_code, 1, 4)  in ('0706','0709','4560','4561','4562','5722','5723','5724','5728','5733','5734','5738','5739','V427') then 1
  when SUBSTR(icd9_code, 1, 3)  in ('570','571') then 1
  else 0 end as liver     /* Liver disease */

, CASE
  when SUBSTR(icd9_code, 1, 4)  in ('2030','2386') then 1
  when SUBSTR(icd9_code, 1, 3)  in ('200','201','202') then 1
  else 0 end as lymph     /* Lymphoma */

, CASE
  when SUBSTR(icd9_code, 1, 3)  in ('196','197','198','199') then 1
  else 0 end as mets      /* Metastatic cancer */

, CASE
  when SUBSTR(icd9_code, 1, 4)  in ('2652','2911','2912','2913','2915','2918','2919','3030','3039','3050','3575','4255','5353','5710','5711','5712','5713','V113') then 1
  when SUBSTR(icd9_code, 1, 3)  in ('980') then 1
  else 0 end as alcohol /* Alcohol abuse */

, CASE
  when icd9_code in ('V6542') then 1
  when SUBSTR(icd9_code, 1, 4)  in ('3052','3053','3054','3055','3056','3057','3058','3059') then 1
  when SUBSTR(icd9_code, 1, 3)  in ('292','304') then 1
  else 0 end as drug /* Drug abuse */

from icd
)

-- collapse the icd9_code specific flags into hadm_id specific flags
-- this groups comorbidities together for a single patient admission

, grp as
(
  select hadm_id
  , max(chf) as chf
  , max(lung) as lung
  , max(dm) as dm
  , max(ren) as ren
  , max(liver) as liver
  , max(lymph) as lymph
  , max(mets) as mets
  , max(alcohol) as alcohol
  , max(drug) as drug
from flag
group by hadm_id
)



select hadm_id
, chf as CONGESTIVE_HEART_FAILURE
, lung as CHRONIC_PULMONARY
, dm as DIABETES
, ren as RENAL_FAILURE
, liver as LIVER_DISEASE
, lymph as LYMPHOMA
, mets as METASTATIC_CANCER
, alcohol as ALCOHOL_ABUSE
, drug as DRUG_ABUSE
, chf+lung+dm+ren+liver+lymph+mets+alcohol+drug as comorbidity
from grp
order by hadm_id;

"""

comorbidity  = run_query(query)
comorbidity.head(10)

,hadm_id,CONGESTIVE_HEART_FAILURE,CHRONIC_PULMONARY,DIABETES,RENAL_FAILURE,LIVER_DISEASE,LYMPHOMA,METASTATIC_CANCER,ALCOHOL_ABUSE,DRUG_ABUSE,comorbidity
0,100001,0,0,1,1,0,0,0,0,0,2
1,100003,0,0,0,0,1,0,0,0,0,1
2,100006,0,0,0,0,0,1,0,0,0,1
3,100007,0,0,0,0,0,0,0,0,0,0
4,100009,0,0,1,0,0,0,0,0,0,1
5,100010,0,0,0,0,0,0,1,0,0,1
6,100011,0,0,0,0,0,0,0,0,1,1
7,100012,0,0,0,0,0,0,0,0,0,0
8,100014,0,0,0,0,0,0,0,0,0,0
9,100016,0,0,0,0,0,0,0,0,0,0


Let's keep only the total score value...

In [ ]:
comorbidity=comorbidity[['hadm_id','comorbidity']]
comorbidity.head(10)

,hadm_id,comorbidity
0,100001,2
1,100003,1
2,100006,1
3,100007,0
4,100009,1
5,100010,1
6,100011,1
7,100012,0
8,100014,0
9,100016,0


# Patient outcome: hospital mortality and mortality at 90 days

We'll use patient parameters on day 1 of ICU admission to predict whether they'll be dead or alive at the end of their hospital stay and 90 days after ICU admission. Note: if the patients dies after leaving the hospital, but before 90 days, his hospital mortality will be 0 but his 90-day mortality will be 1.

In [ ]:
query="""
select ad.hadm_id
    , i.icustay_id
    , admittime
    , dischtime
    , ROW_NUMBER() over (partition by ad.subject_id order by i.intime asc) as adm_order
    , dob
    , dod
    , p.expire_flag as died
    , ad.hospital_expire_flag as morta_hosp
    , case when (datetime_diff(dod,admittime,day)<=90) = True then 1 else 0 end as morta_90d
from `mindstream-ai.mimiciii_clinical.admissions` ad, `mindstream-ai.mimiciii_clinical.icustays` i, `mindstream-ai.mimiciii_clinical.patients` p
where ad.hadm_id=i.hadm_id and p.subject_id=i.subject_id
order by ad.hadm_id asc, adm_order
"""
d  = run_query(query)
d.head()

,hadm_id,icustay_id,admittime,dischtime,adm_order,dob,dod,died,morta_hosp,morta_90d
0,100001,275225,2117-09-11 11:46:00,2117-09-17 16:45:00,1,2082-03-21,NaT,0,0,0
1,100003,209281,2150-04-17 15:34:00,2150-04-21 17:30:00,1,2090-05-19,2150-12-28,1,0,0
2,100006,291788,2108-04-06 15:49:00,2108-04-18 17:18:00,1,2059-05-07,2109-10-24,1,0,0
3,100007,217937,2145-03-31 05:33:00,2145-04-07 12:40:00,1,2071-06-04,NaT,0,0,0
4,100009,253656,2162-05-16 15:56:00,2162-05-21 13:37:00,1,2101-07-30,NaT,0,0,0


### Exercise: extract only hadm_id, hospital and 90-day mortality from the table above, and only for the first ICU stay in each hospital stay

In [ ]:
query="""
with t1 as (
select ad.hadm_id
    , i.icustay_id
    , admittime
    , dischtime
    , ROW_NUMBER() over (partition by ad.subject_id order by i.intime asc) as adm_order
    , dob
    , dod
    , p.expire_flag as died
    , ad.hospital_expire_flag as morta_hosp
    , case when (datetime_diff(dod,admittime,day)<=90) = True then 1 else 0 end as morta_90d
from `mindstream-ai.mimiciii_clinical.admissions` ad, `mindstream-ai.mimiciii_clinical.icustays` i, `mindstream-ai.mimiciii_clinical.patients` p
where ad.hadm_id=i.hadm_id and p.subject_id=i.subject_id
order by ad.hadm_id asc, adm_order
)

select hadm_id,morta_hosp, morta_90d
from t1
where adm_order=1
"""
mortality = run_query(query)
mortality.head()

,hadm_id,morta_hosp,morta_90d
0,100001,0,0
1,100003,0,0
2,100006,0,0
3,100007,0,0
4,100009,0,0


In [ ]:
# average mortality
mortality.morta_90d.mean()

0.15119631637834582

---

# DATA MERGING

Here, we'll **join** all the subtables we have created along the way into a single big dataframe.

In [ ]:
d=pd.merge(vitals, cohort, how='left', on='stay_id')
d = pd.merge(d,labs,how='left',on=('stay_id', 'offset'))
d = pd.merge(d,uo,how='left',on=('stay_id', 'offset'))
d.head(100)

,stay_id,chartTime_x,offset,HR,MBP,SpO2,RR,subject_id,hadm_id,icu_admittime,age,suspected_infection_time,left_window,chartTime_y,Na,Hb,uo,cum_uo
0,32610785,2112-12-02 13:00:00,-24.0,NaN,NaN,95.0,23.0,10002348,22725460,2112-11-30 23:24:00,77,2112-12-03 13:46:00,2112-12-03 13:46:00,NaT,NaN,NaN,NaN,NaN
1,32610785,2112-12-02 14:03:00,-23.0,NaN,102.0,NaN,NaN,10002348,22725460,2112-11-30 23:24:00,77,2112-12-03 13:46:00,2112-12-03 13:46:00,NaT,NaN,NaN,NaN,NaN
2,32610785,2112-12-02 16:19:00,-21.0,63.0,83.0,90.0,18.0,10002348,22725460,2112-11-30 23:24:00,77,2112-12-03 13:46:00,2112-12-03 13:46:00,NaT,NaN,NaN,100.0,475.0
3,32610785,2112-12-02 18:35:00,-19.0,NaN,83.0,NaN,NaN,10002348,22725460,2112-11-30 23:24:00,77,2112-12-03 13:46:00,2112-12-03 13:46:00,NaT,NaN,NaN,NaN,NaN
4,32610785,2112-12-02 22:00:00,-15.0,63.0,NaN,93.0,NaN,10002348,22725460,2112-11-30 23:24:00,77,2112-12-03 13:46:00,2112-12-03 13:46:00,NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,33987268,2156-04-12 21:00:00,5.0,NaN,80.0,98.0,29.0,10002428,28662225,2156-04-12 16:24:18,80,2156-04-12 10:20:00,2156-04-12 16:24:18,NaT,NaN,NaN,80.0,205.0
96,33987268,2156-04-12 22:00:00,6.0,117.0,NaN,NaN,NaN,10002428,28662225,2156-04-12 16:24:18,80,2156-04-12 10:20:00,2156-04-12 16:24:18,NaT,NaN,NaN,110.0,315.0
97,33987268,2156-04-13 00:00:00,8.0,117.0,NaN,96.0,NaN,10002428,28662225,2156-04-12 16:24:18,80,2156-04-12 10:20:00,2156-04-12 16:24:18,NaT,NaN,NaN,25.0,380.0
98,33987268,2156-04-13 01:00:00,9.0,NaN,60.0,NaN,NaN,10002428,28662225,2156-04-12 16:24:18,80,2156-04-12 10:20:00,2156-04-12 16:24:18,2156-04-13 01:41:00,128.0,10.2,30.0,410.0


Remove unwanted columns!


In [ ]:
d = pd.merge(cohort, gender, how='left',on='stay_id')
d = pd.merge(d,comorbidity,how='left',on='hadm_id')
d = pd.merge(d,vitals,how='left',on='hadm_id')
d = pd.merge(d,uo,how='left',on='hadm_id')
d = pd.merge(d,labs,how='left',on='hadm_id')
data = pd.merge(d,mortality,how='left',on='hadm_id')

data.head(2)

,hadm_id,age,gender,comorbidity,weight,gcs,hr,mbp,rr,spo2,tempc,cvp,urineoutput,sodium,glucose,urea,creatinine,bilirubin,albumin,hb,wbc,crp,ph,po2,pco2,morta_hosp,morta_90d
0,100001,35,0,2.0,NaN,NaN,124.0,75.0,22.0,96.0,37.77,NaN,3900.0,144.0,185.0,42.0,2.4,NaN,NaN,11.0,11.2,NaN,NaN,NaN,NaN,0.0,0.0
1,100003,60,1,1.0,85.0,NaN,104.0,47.0,21.0,87.0,36.77,7.0,2580.0,133.0,113.0,49.0,1.2,5.5,2.3,7.1,14.2,NaN,7.37,87.0,29.0,0.0,0.0


# DATA EXPLORATION

### Missing data

Let's look at the count of missing values per column.

In [ ]:
np.sum(np.isnan(d))

stay_id                       0
chartTime_x                   0
offset                        0
HR                           53
MBP                          42
SpO2                         57
RR                           58
subject_id                    0
hadm_id                       0
icu_admittime                 0
age                           0
suspected_infection_time      0
left_window                   0
chartTime_y                 114
Na                          115
Hb                          116
dtype: int64

# Save dataset locally for future sessions

In [ ]:
# download the dataset locally
data.to_csv('data.csv')
files.download('data.csv')

---